# Models released in September 2026: the same layer, the same failure

Causilo (3rd on TabArena, 28.9.2026) and LimiX-2 (1st) through the measurements that carry
the thesis, next to the models measured before. One ensemble member everywhere; the
drivers are `run/new_models.sh` and `experiments/side/head_grid_limix.py`. Files are read
if present, so a partial run prints what exists and names what is missing.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np, pandas as pd
from common import paths, tables, stats, quiet
quiet.silence()
pd.set_option("display.width", 200)
R = lambda n: pd.read_csv(paths.result(n))
NEW = ["causilo", "limix2", "tabpfn35"]


def with_new(base, stem, sub):
    """The measured file(s) plus every new model's file of the same experiment."""
    frames = [R(f"{sub}/{b}") for b in base]
    for tag in NEW:
        p = paths.ROOT / "results" / sub / f"{stem}_{tag}.csv"
        if p.exists():
            frames.append(pd.read_csv(p))
        else:
            print(f"  (missing: {p.name})")
    return tables.ok_rows(pd.concat(frames, ignore_index=True))


def arm_c(d):
    return np.where(d.arm == "clip", "clip_" + d.clip_c.map("{:g}".format), d.arm)

## 1. Natural extremes and capping on freMTPL2sev

The subsamples of Figures 2 and 4 of the note: seed 31337, five per leverage bin. The ratio
is Q(0.99) with the largest claim against without it, on the subsamples where the claim at
least quadruples the sd. Capping is `clip_200` against the raw context on the ten
subsamples with an sd shift of 2 or more.

In [2]:
d = with_new(["clip_context_real.csv", "clip_context_real_v25v26.csv"],
             "clip_context_real", "h3_repair")
d = d.drop_duplicates(["dataset", "model", "bin", "repeat", "arm", "clip_c", "n_est",
                       "n_fit", "n_test"], keep="last")
d["arm_c"] = arm_c(d)
x = d[(d.arm_c == "raw") & (d.bin == "4.0-inf")]
print("Q(0.99) ratio with / without the largest claim, sd shift >= 4")
print(x.groupby("model").d_q99.agg(median=lambda s: np.median(1 + s), low=lambda s: 1 + s.min(),
                                   high=lambda s: 1 + s.max(), n="count")
      .sort_values("median").round(2).to_string())
print()
idx = ["dataset", "model", "bin", "repeat", "n_est", "n_fit", "n_test"]
raw = d[d.arm_c == "raw"].set_index(idx)
for arm in ["clip_200", "clip_50", "sd_cap"]:
    cap = d[(d.arm_c == arm) & d.bin.isin(["2.0-4.0", "4.0-inf"])].set_index(idx)
    rows = []
    for lv in ["pb99", "pb999"]:
        rel = (cap[lv] / raw[lv].reindex(cap.index) - 1).groupby("model")
        rows.append(pd.DataFrame({f"{lv} median": rel.median(),
                                  f"{lv} lower in": rel.apply(lambda s: f"{(s < 0).sum()}/{len(s)}")}))
    print(f"{arm} against raw, sd shift >= 2")
    print(pd.concat(rows, axis=1).to_string(float_format=lambda v: f"{v:+.3f}"))
    print()

  (missing: clip_context_real_tabpfn35.csv)
Q(0.99) ratio with / without the largest claim, sd shift >= 4
             median   low  high  n
model                             
TabPFN-v2.5    0.48  0.48  0.59  5
TabPFN-v2.6    0.51  0.35  0.82  5
TabICLv2       0.85  0.72  0.87  5
TabPFN-V3      0.92  0.80  0.99  5
GBM            0.98  0.93  1.17  5
XGB            0.99  0.83  1.14  5
EXAONE         1.01  0.94  1.04  5
CB             1.04  0.95  1.15  5
LimiX-2        1.25  1.13  1.39  5
Causilo        1.27  0.99  1.38  5
TabDPT         1.67  1.05  2.65  5

clip_200 against raw, sd shift >= 2
             pb99 median pb99 lower in  pb999 median pb999 lower in
model                                                              
CB                -0.693         10/10        -0.762          10/10
Causilo           -0.048          9/10        +0.092           1/10
EXAONE            -0.018         10/10        +0.064           1/10
GBM               -0.097         10/10        -0.012          

clip_50 against raw, sd shift >= 2
             pb99 median pb99 lower in  pb999 median pb999 lower in
model                                                              
CB                -0.763         10/10        -0.826          10/10
Causilo           -0.038          8/10        +0.307           0/10
EXAONE            -0.017          8/10        +0.172           1/10
GBM               -0.111         10/10        +0.374           0/10
LimiX-2           -0.121         10/10        +0.090           4/10
TabDPT            -0.055          8/10        +0.375           1/10
TabICLv2          -0.091          8/10        +0.295           0/10
TabPFN-V3         -0.047         10/10        +0.292           0/10
TabPFN-v2.5       -0.089          9/10        +0.410           1/10
TabPFN-v2.6       -0.118         10/10        +0.333           3/10
XGB               -0.151         10/10        +0.245           1/10

sd_cap against raw, sd shift >= 2
             pb99 median pb99 lower in  pb999 

## 2. One row dosed on the maximum

`y0 = 100 * max(y_train)` at the centre of feature space, five seeds. EXAONE was run at
four members (`influence_exaone.csv`) and is shown as measured.

In [3]:
inf = with_new(["influence.csv", "influence_exaone.csv"], "influence", "h2_leverage")
for xi in (0.7, 0.3):
    top = inf[(inf.xi == xi) & (inf.dose == 100)]
    g = top.groupby(["model", "n_est"])
    print(f"xi = {xi}, dose 100 (median sd shift {top.sd_shift.median():.0f})")
    print(pd.DataFrame({"q99 median": g.local_q99.median(), "q99 min": g.local_q99.min(),
                        "q99 max": g.local_q99.max(), "q50 median": g.local_q50.median(),
                        "dxi median": g.apply(lambda s: (s.xi_with - s.xi_without).median()),
                        "n": g.size()}).round(3).to_string())
    print()

  (missing: influence_tabpfn35.csv)
xi = 0.7, dose 100 (median sd shift 71)
                 q99 median  q99 min  q99 max  q50 median  dxi median  n
model     n_est                                                         
Causilo   1           0.227   -0.144    0.535      -0.025       0.228  5
EXAONE    4           0.208   -0.006    1.807       1.232      -0.250  5
GBM       1           0.096   -0.032    0.350      -0.001       0.005  5
LimiX-2   1           1.072    0.375    5.454       1.586      -0.596  5
TabICLv2  1          -0.490   -0.614   -0.380       0.256      -0.092  5
TabPFN-V3 1          -0.137   -0.549    2.612       1.854      -0.829  5

xi = 0.3, dose 100 (median sd shift 38)
                 q99 median  q99 min  q99 max  q50 median  dxi median  n
model     n_est                                                         
Causilo   1           0.052   -0.057    0.130       0.079       0.004  5
EXAONE    4           0.007   -0.023    0.074       0.198       0.036  5
GBM    

## 3. One row dosed on the sd shift

Implied tail index of the predictive distribution (median `xi_with` over five seeds) and
the clean-context value it starts from. The TabPFN-V3 row is recomputed from the current
`influence_sd.csv`; FINDINGS.md quoted an earlier version of that file.

In [4]:
sd = with_new(["influence_sd.csv"], "influence_sd", "h2_leverage")
for xi in sorted(sd.xi.unique()):
    g = sd[sd.xi == xi]
    tab = g.pivot_table(index="model", columns="dose", values="xi_with", aggfunc="median")
    tab.insert(0, "clean", g.groupby("model").xi_without.median())
    print(f"true xi = {xi}: implied xi")
    print(tab.round(3).to_string())
    print("  local Q(0.5), median change:")
    print(g.pivot_table(index="model", columns="dose", values="local_q50",
                        aggfunc="median").round(3).to_string())
    print("  local Q(0.99), median change:")
    print(g.pivot_table(index="model", columns="dose", values="local_q99",
                        aggfunc="median").round(3).to_string())
    print()

  (missing: influence_sd_tabpfn35.csv)
true xi = 0.3: implied xi
dose       clean    2.0    4.0   10.0   20.0   50.0
model                                              
Causilo    0.391  0.340  0.351  0.360  0.388  0.410
GBM        0.318  0.373  0.373  0.373  0.373  0.373
LimiX-2    0.294  0.288  0.278  0.274  0.239  0.104
TabICLv2   0.448  0.469  0.304  0.278  0.296  0.317
TabPFN-V3  0.330  0.306  0.266  0.123  0.177 -0.173
  local Q(0.5), median change:
dose        2.0    4.0    10.0   20.0   50.0
model                                       
Causilo    0.122  0.173  0.150  0.096  0.091
GBM        0.000  0.000  0.000  0.000  0.000
LimiX-2    0.024  0.137  0.196  0.370  0.714
TabICLv2   0.097  0.140  0.115  0.108  0.103
TabPFN-V3  0.012  0.071  0.250  0.404  0.741
  local Q(0.99), median change:
dose        2.0    4.0    10.0   20.0   50.0
model                                       
Causilo    0.041  0.052  0.052  0.056  0.079
GBM        0.031  0.031  0.031  0.031  0.031
LimiX-2    0.

## 4. Shape against scale on the generator

One ensemble member, twenty seeds, the table of H1. The seed is the free unit.

In [5]:
shape = with_new(["shape_of_x_nest1.csv", "shape_of_x_exaone_nest1.csv"],
                 "shape_of_x", "h1_shape_vs_scale")
scale = with_new(["scale_of_x.csv", "scale_of_x_exaone_nest1.csv"],
                 "scale_of_x", "h1_shape_vs_scale")
tab = pd.DataFrame({"scale share": tables.scale_share_by_model(scale).share,
                    "shape share": tables.slope_by_model(shape).share})
print(tab.sort_values("scale share", ascending=False).round(3).to_string())
print("medians over models: scale %.0f%%, shape %.0f%%"
      % (100 * tab["scale share"].median(), 100 * tab["shape share"].median()))
print()
runs = tables.share_by_run(shape, scale)
for m, g in runs.groupby("model"):
    r = stats.paired_by_cluster(g, "scale_share", "shape_share", cluster="seed")
    print("  %-10s scale closer to 1 than shape in %d of %d seeds, p = %.2g"
          % (m, r["cluster_wins"], r["clusters"], r["cluster_p"]))

  (missing: shape_of_x_tabpfn35.csv)


  (missing: scale_of_x_tabpfn35.csv)
           scale share  shape share
model                              
TabICLv2         0.984        0.444
TabPFN-V3        0.952        0.425
LimiX-2          0.930        0.372
Causilo          0.928        0.461
EXAONE           0.927        0.368
GBM              0.872        0.453
medians over models: scale 93%, shape 43%



  Causilo    scale closer to 1 than shape in 19 of 20 seeds, p = 2e-05
  EXAONE     scale closer to 1 than shape in 20 of 20 seeds, p = 9.5e-07
  GBM        scale closer to 1 than shape in 19 of 20 seeds, p = 2e-05
  LimiX-2    scale closer to 1 than shape in 19 of 20 seeds, p = 2e-05
  TabICLv2   scale closer to 1 than shape in 20 of 20 seeds, p = 9.5e-07
  TabPFN-V3  scale closer to 1 than shape in 20 of 20 seeds, p = 9.5e-07


## 5. A unit error on seven tables

One context target times 100, ten subsamples per table, the corrupted row drawn among those
whose error reaches an sd shift of 2. The new models ran four arms (raw, clip_50, clip_200,
sd_cap). Medians are over runs, as in the note's reference table.

In [6]:
u = with_new(["unit_error_real.csv"], "unit_error_real", "h3_repair")
u["arm_c"] = arm_c(u)
idx = ["dataset", "model", "repeat"]
clean = u[(u.arm_c == "raw") & (u.factor == 1)].set_index(idx)
err = u[(u.arm_c == "raw") & (u.factor == 100)].set_index(idx)
cap = u[(u.arm_c == "clip_200") & (u.factor == 100)].set_index(idx)


def summary(models):
    c = clean[clean.index.get_level_values("model").isin(models)]
    e = err.reindex(c.index).dropna(subset=["pb99"])
    k = cap.reindex(e.index)
    r999 = k.pb999 / e.pb999 - 1
    return {"runs": len(e), "0.99 error vs clean": (e.pb99 / c.pb99.reindex(e.index) - 1).median(),
            "0.99 capped error vs clean": (k.pb99 / c.pb99.reindex(e.index) - 1).median(),
            "0.999 capped vs error": r999.median(), "capping doubled 0.999": int((r999 > 1).sum())}


groups = {"note: v2.5, v2.6, V3, TabICLv2": ["TabPFN-v2.5", "TabPFN-v2.6", "TabPFN-V3", "TabICLv2"]}
groups.update({m: [m] for m in ["TabPFN-V3", "TabICLv2", "Causilo", "LimiX-2", "TabPFN-v3.5", "GBM"]
               if m in set(u.model)})
pct = lambda v: f"{v:+.2%}"
fmt = {"runs": "{:.0f}".format, "capping doubled 0.999": "{:.0f}".format,
       "0.99 error vs clean": pct, "0.99 capped error vs clean": pct, "0.999 capped vs error": pct}
print("all tables each model has")
print(pd.DataFrame({k: summary(v) for k, v in groups.items()}).T.to_string(formatters=fmt))
done = err.reset_index().groupby("model").dataset.unique()
shared = sorted(set.intersection(*[set(done[m]) for g in groups.values() for m in g]))
print()
print(f"the {len(shared)} tables every model has: {', '.join(shared)}")
full = (clean, err, cap)
clean, err, cap = (t[t.index.get_level_values("dataset").isin(shared)] for t in full)
print(pd.DataFrame({k: summary(v) for k, v in groups.items()}).T.to_string(formatters=fmt))
clean, err, cap = full

  (missing: unit_error_real_tabpfn35.csv)
all tables each model has
                               runs 0.99 error vs clean 0.99 capped error vs clean 0.999 capped vs error capping doubled 0.999
note: v2.5, v2.6, V3, TabICLv2  280              +2.06%                     +0.54%                +0.18%                     2
TabPFN-V3                        70              +2.05%                     +1.06%                +0.31%                     0
TabICLv2                         70              +4.10%                     +4.26%                -1.30%                     0
Causilo                          70              +2.69%                     +1.86%                -0.27%                     0
LimiX-2                          70             +14.01%                     +3.71%               -13.00%                     0
GBM                              70              +8.02%                     +1.29%                -3.80%                     0

the 7 tables every model has: AutoClaims, 